In [ ]:
%pip install datasets pyyaml torchmetrics tokenizers tqdm tensorboard wandb

In [91]:
import warnings
import os
import torch
import torch.nn
from torch.utils.data import Dataset, DataLoader, random_split

import tqdm
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.trainers import WordLevelTrainer
from tokenizers.pre_tokenizers import Whitespace

from pathlib import Path
from dataset import EngVietDataset, casual_mask
from model import build_transformer

from config import get_config, update_config, get_weights_path, get_latest_weight

import torchmetrics

from torch.utils.tensorboard import SummaryWriter
from torch.optim.lr_scheduler import LambdaLR

In [92]:
def get_all_sentences(dataset, lang):
  for item in dataset:
    yield item['translation'][lang]

In [93]:
def build_tokenizer(config, dataset, lang):
  # config
  tokenizer_path = Path(config['tokenizer_file'].format(lang))
  if not Path.exists(tokenizer_path):
    tokenizer = Tokenizer(WordLevel(unk_token='[UNKNOWN]'))
    tokenizer.pre_tokenizer = Whitespace()
    trainer = WordLevelTrainer(special_tokens=["[UNKNOWN]", "[PADDING]", "[SOS]", "[EOS]"], min_frequency=2)
    tokenizer.train_from_iterator(get_all_sentences(dataset, lang), trainer=trainer)
  else:
    tokenizer = Tokenizer.from_file(str(tokenizer_path))
  return tokenizer

In [94]:
def get_dataset(config):
  train_test_dataset_raw = load_dataset(f"{config['datasource']}")
  train_raw = train_test_dataset_raw['train']
  #test_raw = train_test_dataset_raw['test']

  #Build tokenizers
  tokenizer_src = build_tokenizer(config, train_raw, config['lang_src'])
  tokenizer_tgt = build_tokenizer(config, train_raw, config['lang_tgt'])
  
  #Spilt to train - validation 90-10
  train_ds_size = int(0.9 * len(train_raw))
  val_ds_size = len(train_raw) - train_ds_size

  train_ds_raw, val_ds_raw = random_split(train_raw, [train_ds_size, val_ds_size])

  train_ds = EngVietDataset(train_ds_raw, tokenizer_src, tokenizer_tgt, config['lang_src'], config['lang_tgt'], config['seq_len'])
  val_ds = EngVietDataset(val_ds_raw, tokenizer_src, tokenizer_tgt, config['lang_src'], config['lang_tgt'], config['seq_len'])

  max_len_src = 0
  max_len_tgt = 0

  for item in train_raw:
    src_ids = tokenizer_src.encode(item['translation'][config['lang_src']]).ids
    tgt_ids = tokenizer_tgt.encode(item['translation'][config['lang_tgt']]).ids
    max_len_src = max(max_len_src, len(src_ids))
    max_len_tgt = max(max_len_tgt, len(tgt_ids))

  print(f'Max length of source sentence: {max_len_src}')
  print(f'Max length of source sentence: {max_len_tgt}')

  update_config({"max_len_src" : max_len_src , "max_len_tgt" : max_len_tgt})

  train_data_loader = DataLoader(train_ds, batch_size=config['batch_size'], shuffle=True)
  val_data_loader = DataLoader(val_ds, batch_size=config['batch_size'], shuffle=True)

  return train_data_loader, val_data_loader, tokenizer_src, tokenizer_tgt

In [95]:
def get_model(config, vocab_src_len, vocab_tgt_len):
  return build_transformer(vocab_src_len
                           , vocab_tgt_len
                           , config["seq_len"]
                           , config["seq_len"]
                           , d_model=config['d_model']
                           , N=config['N']
                           , num_heads=config['num_heads']
                           , dropout=config['dropout']
                           , d_ff=config['d_ff'])

In [ ]:
config = get_config()
config

In [97]:
def greedy_decode(model, source, source_mask, tokenizer_src, tokenizer_tgt, max_len, device):
  sos_idx = tokenizer_tgt.token_to_id('[SOS]')
  eos_idx = tokenizer_tgt.token_to_id('[EOS]')

  # Precompute the encoder output and reuse it for every step
  encoder_output = model.encode(source, source_mask)
  # Initialize the decoder input with the sos token
  decoder_input = torch.empty(1, 1).fill_(sos_idx).type_as(source).to(device)
  while True:
    if decoder_input.size(1) == max_len:
      break

    # build mask for target
    decoder_mask = casual_mask(decoder_input.size(1)).type_as(source_mask).to(device)

    # calculate output
    out = model.decode(encoder_output, source_mask, decoder_input, decoder_mask)

    # get next token
    prob = model.project(out[:, -1])
    _, next_word = torch.max(prob, dim=1)
    decoder_input = torch.cat(
      [decoder_input, torch.empty(1, 1).type_as(source).fill_(next_word.item()).to(device)], dim=1
    )

    if next_word == eos_idx:
      break

  return decoder_input.squeeze(0)

In [98]:
def run_validation(model, validation_ds, tokenizer_src, tokenizer_tgt, max_len, device, print_msg, global_step, writer, num_examples=2):
    model.eval()
    count = 0

    source_texts = []
    expected = []
    predicted = []

    try:
        # get the console window width
        with os.popen('stty size', 'r') as console:
            _, console_width = console.read().split()
            console_width = int(console_width)
    except:
        # If we can't get the console width, use 80 as default
        console_width = 80

    with torch.no_grad():
        for batch in validation_ds:
            count += 1
            encoder_input = batch["encoder_input"].to(device) # (b, seq_len)
            encoder_mask = batch["encoder_mask"].to(device) # (b, 1, 1, seq_len)

            # check that the batch size is 1
            assert encoder_input.size(
                0) == 1, "Batch size must be 1 for validation"

            model_out = greedy_decode(model, encoder_input, encoder_mask, tokenizer_src, tokenizer_tgt, max_len, device)

            source_text = batch["src_text"][0]
            target_text = batch["tgt_text"][0]
            model_out_text = tokenizer_tgt.decode(model_out.detach().cpu().numpy())

            source_texts.append(source_text)
            expected.append(target_text)
            predicted.append(model_out_text)

            # Print the source, target and model output
            print_msg('-'*console_width)
            print_msg(f"{f'SOURCE: ':>12}{source_text}")
            print_msg(f"{f'TARGET: ':>12}{target_text}")
            print_msg(f"{f'PREDICTED: ':>12}{model_out_text}")

            if count == num_examples:
                print_msg('-'*console_width)
                break

    if writer:
        metric = torchmetrics.CharErrorRate()
        cer = metric(predicted, expected)
        writer.add_scalar('validation cer', cer, global_step)
        writer.flush()
        
        metric = torchmetrics.WordErrorRate()
        wer = metric(predicted, expected)
        writer.add_scalar('validation wer', wer, global_step)
        writer.flush()
        
        metric = torchmetrics.BLEUScore()
        bleu = metric(predicted, expected)
        writer.add_scalar('validation BLEU', bleu, global_step)
        writer.flush()
    

In [119]:
def train_model(config):
    # Determine the available device
  if torch.cuda.is_available():
    device = "cuda"
  elif torch.backends.mps.is_available():
    device = "mps"  # MPS for macOS with Metal Performance Shaders
  else:
    device = "cpu"

  print("Using device:", device)

  # Display device details
  if device == "cuda":
    print(f"Device name: {torch.cuda.get_device_name(torch.cuda.current_device())}")
    print(f"Device memory: {torch.cuda.get_device_properties(torch.cuda.current_device()).total_memory / 1024 ** 3:.2f} GB")
  elif device == "mps":
    print("Device name: Apple Silicon (MPS)")
    print("Note: PyTorch on MPS is experimental and may have limitations.")
  else:
    print("Device name: CPU")


  device = torch.device(device)

  Path(f"{config['datasource']}_{config['model_folder']}").mkdir(parents=True,exist_ok=True)

  #Dataset
  train_data_loader, val_data_loader, tokenizer_src, tokenizer_tgt = get_dataset(config)

  #Model
  model = get_model(config,
                    tokenizer_src.get_vocab_size(),
                    tokenizer_tgt.get_vocab_size(),
                    ).to(device)

  #Tensor board
  writer = SummaryWriter(config['experiment_name'])

  #Adam
  optimizer = torch.optim.Adam(model.parameters(), lr=float(config['lr']), betas=(config['B1'],config['B2']), eps=float(config['ep']))

  #Preloading if specified
  initial_epoch = 0
  global_step = 0
  preload = config['preload']
  model_filename = get_latest_weight(config) if preload == 'latest' else get_weights_path(config,initial_epoch) if preload != 'None' else None

  if model_filename is not None and Path.exists(model_filename):
    print(f'Preloading model {model_filename}')
    state = torch.load(model_filename)
    model.load_state_dict(state['model_state_dict'])
    initial_epoch = state['epoch'] + 1
    optimizer.load_state_dict(state['optimizer_state_dict'])
    global_step = state['global_step']
  else:
    print('No model to preload, starting from scratch')
    
  # CrossEntropyLoss with label smoothing
  loss_fn = torch.nn.CrossEntropyLoss(ignore_index=tokenizer_src.token_to_id('[PAD]'), label_smoothing=config['label_smoothing']).to(device)

  for epoch in range(initial_epoch, config['num_epochs']):
        torch.cuda.empty_cache()
        model.train()
        batch_iterator = tqdm(train_data_loader, desc=f"Processing Epoch {epoch:02d}")
        for batch in batch_iterator:

            encoder_input = batch['encoder_input'].to(device) # (b, seq_len)
            decoder_input = batch['decoder_input'].to(device) # (B, seq_len)
            encoder_mask = batch['encoder_mask'].to(device) # (B, 1, 1, seq_len)
            decoder_mask = batch['decoder_mask'].to(device) # (B, 1, seq_len, seq_len)

            # Run the tensors through the encoder, decoder and the projection layer
            encoder_output = model.encode(encoder_input, encoder_mask) # (B, seq_len, d_model)
            decoder_output = model.decode(encoder_output, encoder_mask, decoder_input, decoder_mask) # (B, seq_len, d_model)
            proj_output = model.project(decoder_output) # (B, seq_len, vocab_size)

            # Compare the output with the label
            label = batch['label'].to(device) # (B, seq_len)

            # Compute the loss using a simple cross entropy
            loss = loss_fn(proj_output.view(-1, tokenizer_tgt.get_vocab_size()), label.view(-1))
            batch_iterator.set_postfix({"loss": f"{loss.item():6.3f}"})

            # Log the loss
            writer.add_scalar('train loss', loss.item(), global_step)
            writer.flush()

            # Backpropagate the loss
            loss.backward()

            # Update the weights
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

            global_step += 1

        # Run validation at the end of every epoch
        run_validation(model, val_data_loader, tokenizer_src, tokenizer_tgt, config['seq_len'], device, lambda msg: batch_iterator.write(msg), global_step, writer)

        # Save the model at the end of every epoch
        model_filename = get_weights_path(config, f"{epoch:02d}")
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'global_step': global_step
        }, model_filename)

In [ ]:
warnings.filterwarnings("ignore")
config = get_config()
train_model(config)